# Synthetic Data Loaders & OOD Detection Demo

This notebook demonstrates how to generate and use synthetic Gaussian worlds with controlled representation shift and anisotropic covariance structures for Out-of-Distribution (OOD) detection experiments.

### Mathematical Setup
- **In-Distribution ($P$)**: $\mathcal{N}(\mu, \Sigma)$ where $\mu = 0$ and $\Sigma = \operatorname{diag}(\text{linspace}(0.3, 3.0, d))$.
- **Shifted / OOD Distribution ($Q$)**: $\mathcal{N}(\mu + \Delta, \Sigma)$.
- **Mahalanobis Separation**: $\Delta$ is scaled along a random direction $u \sim \mathcal{N}(0, I)$ such that:
  $$\Delta^T \Sigma^{-1} \Delta = \text{sep}^2$$
- **Mixture Draws**: Samples drawn from $(1 - \pi) P + \pi Q$ with indicator $z \in \{0, 1\}$ ($z=1 \iff x \sim Q$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Import from src.data.synth (or define directly)
from src.data.synth import make_world, draw, splits

# Optional: import detectors if available
try:
    from detectors import Maha, KNN, Linear, Spectral, calibrate
    HAS_DETECTORS = True
except ImportError:
    HAS_DETECTORS = False

print(f"NumPy version: {np.__version__}")
print(f"Detectors available: {HAS_DETECTORS}")

## 1. Inspecting the Synthetic Data Loaders (`synth.py`)

Below are the core synthetic data functions implemented in `src/data/synth.py`:

In [ ]:
# Review the data loader definitions
"""
def make_world(d=20, sep=3.0, seed=0):
    rng = np.random.default_rng(seed)
    mu = np.zeros(d)
    Sigma = np.diag(np.linspace(0.3, 3.0, d))          # anisotropic on purpose
    Sinv = np.linalg.inv(Sigma)
    u = rng.normal(size=d)
    Delta = sep * u / np.sqrt(u.T @ Sinv @ u)          # Delta^T Sigma^-1 Delta == sep**2
    return dict(mu=mu, Sigma=Sigma, Delta=Delta, Sinv=Sinv)

def draw(world, n, pi, rng):
    z = rng.random(n) < pi                            # True -> from Q
    X = rng.multivariate_normal(world["mu"], world["Sigma"], n)
    X[z] += world["Delta"]
    return X, z                                       # z is for analysis only, never for fitting

def splits(world, rng, n_ref=500, n_calib=2000, n_probe=200, n_dev=2000, n_oracle=50_000, n_contam=2000, pi_dev=0.5):
    return dict(
        ref    = draw(world, n_ref,    0.0, rng)[0],
        calib  = draw(world, n_calib,  0.0, rng)[0],
        probe  = draw(world, n_probe,  0.0, rng)[0],
        dev    = draw(world, n_dev,    pi_dev, rng),   # (X_dev, z_dev) mixture pool
        oracle = draw(world, n_oracle, 0.0, rng)[0],
        contam = draw(world, n_contam, 1.0, rng)[0],   # pure Q, to be mixed into ref later
    )
"""
print("Data loader definition loaded.")

## 2. Instantiating a Synthetic World & Verifying Mathematical Properties

In [ ]:
# Instantiate world
d = 20
sep = 3.0
seed = 42

world = make_world(d=d, sep=sep, seed=seed)

# Verify Mahalanobis quadratic form
Delta = world["Delta"]
Sinv = world["Sinv"]
actual_sep_sq = float(Delta.T @ Sinv @ Delta)

print(f"Dimension d: {d}")
print(f"Target sep^2: {sep**2:.4f}")
print(f"Actual Delta^T Sigma^-1 Delta: {actual_sep_sq:.4f}")
print(f"Mahalanobis distance (||Delta||_Sigma): {np.sqrt(actual_sep_sq):.4f}")

assert np.isclose(actual_sep_sq, sep**2), "Separation mismatch!"
print("✓ Mathematical separation constraint verified successfully!")

## 3. Generating Data Splits

In [ ]:
rng = np.random.default_rng(123)

# Generate all splits
data = splits(
    world=world,
    rng=rng,
    n_ref=500,
    n_calib=2000,
    n_probe=200,
    n_dev=2000,
    n_oracle=10_000,
    n_contam=1000,
    pi_dev=0.3  # 30% OOD in dev pool
)

X_ref = data["ref"]
X_calib = data["calib"]
X_probe = data["probe"]
X_dev, z_dev = data["dev"]
X_oracle = data["oracle"]
X_contam = data["contam"]

print(f"X_ref (Clean ID):      shape {X_ref.shape}")
print(f"X_calib (Calibration):  shape {X_calib.shape}")
print(f"X_probe (Probe):        shape {X_probe.shape}")
print(f"X_dev (Dev Pool):       shape {X_dev.shape} | OOD fraction z: {z_dev.mean():.2%}")
print(f"X_oracle (Oracle ID):   shape {X_oracle.shape}")
print(f"X_contam (Pure OOD Q):  shape {X_contam.shape}")

## 4. Visualizing In-Distribution $P$ vs Shifted $Q$

Let's project the representations onto the 2D subspace defined by the shift vector $\Delta$ and the top principal axis of $\Sigma$.

In [ ]:
# Direction 1: Normalized shift direction
v1 = Delta / np.linalg.norm(Delta)

# Direction 2: Orthogonal direction along maximum variance
e_max = np.zeros_like(v1)
e_max[-1] = 1.0  # Largest variance axis in diagonal Sigma
v2 = e_max - (e_max @ v1) * v1
v2 /= np.linalg.norm(v2)

# Project splits
proj_ref = X_ref @ np.column_stack([v1, v2])
proj_contam = X_contam @ np.column_stack([v1, v2])
proj_dev_id = X_dev[~z_dev] @ np.column_stack([v1, v2])
proj_dev_ood = X_dev[z_dev] @ np.column_stack([v1, v2])

plt.figure(figsize=(9, 6), dpi=120)
plt.scatter(proj_ref[:, 0], proj_ref[:, 1], alpha=0.4, label="Reference Clean (P)", color="#1f77b4", s=18)
plt.scatter(proj_contam[:, 0], proj_contam[:, 1], alpha=0.4, label="Contaminant (Q)", color="#d62728", s=18)
plt.scatter(proj_dev_id[:, 0], proj_dev_id[:, 1], alpha=0.5, label="Dev Pool (ID, z=0)", marker="+", color="#2ca02c", s=25)
plt.scatter(proj_dev_ood[:, 0], proj_dev_ood[:, 1], alpha=0.5, label="Dev Pool (OOD, z=1)", marker="x", color="#ff7f0e", s=25)

plt.title(f"Synthetic World Representation Space (d={d}, sep={sep})", fontsize=13, fontweight="bold")
plt.xlabel("Projection along Delta / ||Delta||", fontsize=11)
plt.ylabel("Orthogonal Feature Projection", fontsize=11)
plt.legend(frameon=True, fontsize=9)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Fitting Anomaly Detectors & Calibration

We can now fit anomaly detectors (`Maha`, `KNN`, `Linear`, `Spectral`) on the reference split `X_ref`, calibrate rejection thresholds at $1 - \alpha = 0.95$ on `X_calib`, and evaluate rejection rates and AUROC on the `dev` pool.

In [ ]:
if HAS_DETECTORS:
    detectors_dict = {
        "Mahalanobis": Maha(ridge=1e-6),
        "KNN (k=5)": KNN(k=5),
        "Linear (Signed)": Linear(contaminant=X_contam),
        "Spectral": Spectral(),
    }
    
    plt.figure(figsize=(8, 6), dpi=120)
    
    print(f"{'Detector':<20} | {'Calib Thresh (95%)':<20} | {'Dev ID FPR':<15} | {'Dev OOD TPR':<15} | {'AUROC':<10}")
    print("-" * 88)
    
    for name, det in detectors_dict.items():
        # Calibrate on clean calib split (Protocol A)
        fitted, threshold = calibrate(det, X_ref, X_calib, quantile=0.95)
        
        # Score dev pool
        dev_scores = fitted.score(X_dev)
        rejections = dev_scores > threshold
        
        fpr_id = rejections[~z_dev].mean()
        tpr_ood = rejections[z_dev].mean()
        
        fpr_curve, tpr_curve, _ = roc_curve(z_dev, dev_scores)
        roc_auc = auc(fpr_curve, tpr_curve)
        
        print(f"{name:<20} | {threshold:<20.4f} | {fpr_id:<15.2%} | {tpr_ood:<15.2%} | {roc_auc*100:<9.2f}%")
        plt.plot(fpr_curve, tpr_curve, lw=2, label=f"{name} (AUROC = {roc_auc*100:.1f}%)")
        
    plt.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Random Chance")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate (ID Rejection Rate)", fontsize=11)
    plt.ylabel("True Positive Rate (OOD Detection Rate)", fontsize=11)
    plt.title("ROC Curves on Dev Pool", fontsize=13, fontweight="bold")
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("Detectors module not loaded. Run `pip install -e .` to enable full detector evaluation.")

## 6. Separation Sweep ($	ext{sep} \in [1.0, 5.0]$)

Let's sweep the Mahalanobis separation distance $\text{sep}$ and observe the AUROC curve.

In [ ]:
if HAS_DETECTORS:
    sep_values = np.linspace(0.5, 5.0, 10)
    maha_aurocs = []
    knn_aurocs = []
    
    for s in sep_values:
        w = make_world(d=10, sep=s, seed=0)
        d_split = splits(w, np.random.default_rng(0), n_ref=400, n_dev=1000, pi_dev=0.5)
        
        m = Maha().fit(d_split["ref"])
        scores_m = m.score(d_split["dev"][0])
        fpr, tpr, _ = roc_curve(d_split["dev"][1], scores_m)
        maha_aurocs.append(auc(fpr, tpr))
        
        k = KNN(k=5).fit(d_split["ref"])
        scores_k = k.score(d_split["dev"][0])
        fpr_k, tpr_k, _ = roc_curve(d_split["dev"][1], scores_k)
        knn_aurocs.append(auc(fpr_k, tpr_k))
        
    plt.figure(figsize=(8, 5), dpi=120)
    plt.plot(sep_values, np.array(maha_aurocs) * 100, marker="o", lw=2, label="Mahalanobis Detector")
    plt.plot(sep_values, np.array(knn_aurocs) * 100, marker="s", lw=2, label="k-NN Detector (k=5)")
    plt.title("Detection AUROC vs. Mahalanobis Separation (sep)", fontsize=13, fontweight="bold")
    plt.xlabel("Separation parameter (sep)", fontsize=11)
    plt.ylabel("Dev Pool AUROC (%)", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()